# ♟️ Chess Numba — Classical HCE Texel / SPSA 分區調參工作台

在完成 `evaluation.py` 的 **8 項錯誤修復與順序修正**（`BUG-1..8`、`ORD-1..3`、`KP-DISCONT`）後，本 Notebook 提供**單一 Kernel、免重載資料、免重編譯 Numba** 的互動式區塊調參流程。

### 🛡️ 核心設計：自動尺度對齊 ($K^*$)、模式切換、SF19 混合標籤與記憶體安全
1. **`dataset_epd_wdl_20m.npz`（~19,630,000 局面）無洩漏三切分**：
   - **`~16.6M` Train Pool**（16 GB RAM 自動抽樣 `10M` 防 Swap）：僅供 SPSA mini-batch 梯度估計。
   - **`1,000,000` Dev Split (`rng=0`)**：供 SPSA 過程報告、early stopping 與**自動校準 $K^*$ (`k_factor="auto"`)**，徹底消除固定 `K=1.0` 造成的假性尺度膨脹。
   - **`2,000,000` Unseen Holdout (`rng=42`)**：**完全不進訓練、也不進 early stop，且永遠維持純 WDL 評估**，作為第一道靜態驗收閘門。
2. **可切換獨立驗證集（`USE_PGN_VAL` / `USE_ZURICHESS_VAL`）**：
   - **`USE_PGN_VAL = True`（預設雙重驗證）**：同時以 `2M unseen holdout` + `val_pgn.npz`（1.59M 實戰局面，含 `castling!=0` 易位權桶與校準 $K^*$）把關。
   - **`USE_PGN_VAL = False`（純 Holdout 對照模式）**：略過 `val_pgn.npz` 載入與評估，僅依 `2M unseen holdout (@ K*)` 判定。
   - **`USE_ZURICHESS_VAL`（選用第三平靜仲裁集）**：需要時可設 `True` 載入 `val_zurichess.npz`（1.08M 平靜局面，含 13.53% 易位權）。
3. **可切換 SF19 搜尋評估＋WDL 線性混合標籤（`USE_SF_LABELS` & `SF_LAMBDA`）**：
   - 先在終端機執行 `python -m tuner.data_pipeline.annotate_sf_labels` 產生輕量側車檔 `dataset_epd_wdl_20m.sf19.npz`（~35 MB，支援 `Ctrl+C` 中斷續傳與 `--limit` 子集先行測試）。
   - 開啟 `USE_SF_LABELS = True` 後，Session 會在 `train_idx` 上自動擬合無偏 $K_{	ext{sf}}$，於記憶體動態合成 $y_{	ext{mix}} = (1 - \lambda)\, y_{	ext{wdl}} + \lambda\, \sigma(K_{	ext{sf}} \cdot 	ext{sf\_cp})$（並自動剔除 $|	ext{sf\_cp} - 	ext{sf\_d1\_cp}| > 180$ 的戰術跳分局面），同時保持 `2M unseen holdout` 與 `val_pgn` 為**純 WDL** 不變尺規。
4. **目標引擎選擇 (`TARGET_ENGINE`) 與零重載切換 (`session.reconfigure`)**：
   - `"classical"` 寫入 `chess_engine/classical/constants.py`；`"classical_old"` 寫入 `chess_engine/classical_old/constants.py`，輸出與備份自動隔離至 `tuner/runs/<engine>_<mode>[_sfXX]/`。
   - 修改 `# 0` 開關後重跑 `# 0`，自動複用記憶體中的 20M 資料集（~3 秒切換，防 16 GB 機器 OOM）。
5. **Google Colab + Google Drive 雲端執行與斷線防護（`# 00` & `# 9d`）**：
   - 本機執行自動略過 `# 00`；在 Colab 會自動掛載 Google Drive，將 `MyDrive/chess_numba_data/hce/` 資料集複製到本機 SSD，並把每次 `accept()` / `rollback()` / 候選檔即時備份至 `MyDrive/chess_numba_data/texel_runs/`（支援 `RESUME_FROM_DRIVE = True` 斷線續跑）。

### 🚦 Gate Verdict 驗收標準（已升級 $K^*$ 防膨脹守衛）
- `PASS`：`2M unseen holdout` 在校準尺度 $K^*$ 顯著下降（`delta < 0` 且無尺度扭曲）、`broad-EG (phase<=64)` 未惡化（$z \le 1.0$）；若啟用 `val_pgn`，其 $K^*$ 無退步（$z \le 1.2$，排除 $< 1.2\,	ext{SE}$ 微小統計雜訊）且 `castling!=0` 穩定。
- `CAUTION`：主驗證集改善，但子桶或獨立驗證集微幅偏正（$1.0 < z \le 2.0$）。
- `REJECT`：Holdout 在 $K^*$ 退步、`broad-EG` 顯著惡化（$z > 1.8$），或 `val_pgn @ K*` 顯著惡化（$z > 2.0\sim2.5$）→ **自動跳過不套用**。

In [ ]:
# 00. 【Colab / Google Drive 環境初始化】（本機執行會自動略過，直接保留本機路徑與資料）
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = ("google.colab" in sys.modules) or (os.name != "nt" and Path("/content").is_dir())

# ==============================================================================
# Colab / Google Drive 設定（僅在 Google Colab 環境生效）
# ==============================================================================
REPO_URL = "https://github.com/b08203031/chess_numba.git"
BRANCH = "feat-initial-bitboard-setup"
WORKDIR = Path("/content/chess_numba")

# Google Drive 資料目錄：請先手動將 dataset_epd_wdl_20m.npz 與 val_pgn.npz 上傳到此資料夾
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/chess_numba_data/hce")
# 調參產物 (runs / backups / 最新 constants.py) 會即時備份至此 Drive 目錄，防斷線遺失
DRIVE_BACKUP_DIR = Path("/content/drive/MyDrive/chess_numba_data/texel_runs")
# 若 Colab 中途斷線，重跑想接續 Drive 上最後一次 Accept 的 constants.py，請改為 True
RESUME_FROM_DRIVE = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    print("=== Colab 硬體與記憶體規格 ===")
    subprocess.run(["bash", "-lc", "lscpu | grep -E 'Model name|CPU\\(s\\)' || nproc"], check=False)
    subprocess.run(["free", "-h"], check=False)

    # 1) Clone 或更新 GitHub 程式碼（不用 git clean -fd，避免誤刪已複製的資料與快取）
    if not (WORKDIR / ".git").is_dir():
        print(f"\n[Git] Cloning {REPO_URL} (branch={BRANCH}) -> {WORKDIR} ...")
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(WORKDIR)], check=True)
    else:
        print(f"\n[Git] Syncing {WORKDIR} -> origin/{BRANCH} ...")
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "checkout", "-f", BRANCH], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)

    os.chdir(WORKDIR)

    # 2) 僅安裝 Colab 缺少的必要相依套件（保留預裝相容之 NumPy / Numba / SciPy 組合）
    missing_pkgs = [
        pkg
        for mod, pkg in (
            ("numba", "numba"),
            ("numpy", "numpy"),
            ("scipy", "scipy"),
            ("chess", "chess"),
        )
        if importlib.util.find_spec(mod) is None
    ]
    if missing_pkgs:
        print(f"[Pip] Installing missing packages: {missing_pkgs}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs], check=True)

    # 3) 檢查 / 還原斷線續跑備份
    DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
    if RESUME_FROM_DRIVE:
        restored = []
        for eng in ("classical", "classical_old"):
            saved = DRIVE_BACKUP_DIR / f"{eng}_constants.py"
            dst_c = WORKDIR / "chess_engine" / eng / "constants.py"
            if saved.is_file():
                shutil.copy2(saved, dst_c)
                restored.append(eng)
                print(f"[Resume] 已從 Drive 還原 {eng}/constants.py <- {saved}")
        # 同步先前的 runs 子目錄（含 backups/ 與已產生的 block_*.py）
        for sub in DRIVE_BACKUP_DIR.iterdir():
            if sub.is_dir():
                dst_run = WORKDIR / "tuner" / "runs" / sub.name
                shutil.copytree(sub, dst_run, dirs_exist_ok=True)
                print(f"[Resume] 已還原歷史調參紀錄 -> {dst_run}")
        if restored:
            subprocess.run([sys.executable, "-m", "tuner.sync_constants_snapshot"], check=True)
        if "classical_old" in restored:
            subprocess.run(
                [
                    sys.executable,
                    "-c",
                    "from pathlib import Path; "
                    "from tuner.sync_constants_snapshot import clear_classical_numba_cache; "
                    "clear_classical_numba_cache(Path('chess_engine/classical_old/__pycache__'))",
                ],
                check=True,
            )
    else:
        for eng in ("classical", "classical_old"):
            saved = DRIVE_BACKUP_DIR / f"{eng}_constants.py"
            if saved.is_file():
                print(
                    f"[Info] 偵測到 Drive 上已有備份：{saved}\n"
                    "       目前 RESUME_FROM_DRIVE=False（將從 GitHub 最新基準重新起跑）；"
                    "若要從上次斷線處續跑，請改設 RESUME_FROM_DRIVE=True 後重跑本 Cell。"
                )

    # 4) 從 Google Drive 複製資料集到 Colab 本機 SSD（比直接從 Drive FUSE 讀取快 ~10x）
    local_data_dir = WORKDIR / "tuner" / "data" / "hce"
    local_data_dir.mkdir(parents=True, exist_ok=True)
    if not DRIVE_DATA_DIR.is_dir():
        DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)
        raise FileNotFoundError(
            f"找不到 Drive 資料夾或尚未上傳檔案：{DRIVE_DATA_DIR}\n"
            "請先手動將本機 tuner/data/hce/ 中的檔案上傳至 Google Drive 'MyDrive/chess_numba_data/hce/'：\n"
            "  1) dataset_epd_wdl_20m.npz（必備，~674 MB）\n"
            "  2) val_pgn.npz（開 USE_PGN_VAL=True 時必備，~22.7 MB）\n"
            "  3) val_zurichess.npz / dataset_epd_wdl_20m.sf19.npz（選用）"
        )

    data_files = [
        ("dataset_epd_wdl_20m.npz", True),
        ("val_pgn.npz", False),
        ("val_zurichess.npz", False),
        ("dataset_epd_wdl_20m.sf19.npz", False),
    ]
    for fname, required in data_files:
        src = DRIVE_DATA_DIR / fname
        dst = local_data_dir / fname
        if src.is_file():
            if dst.is_file() and dst.stat().st_size == src.stat().st_size:
                print(f"[Data] 本機 SSD 已存在最新 {fname} ({dst.stat().st_size / (1024**2):.1f} MB)，略過複製。")
            else:
                print(f"[Data] 正在從 Drive 複製 {fname} ({src.stat().st_size / (1024**2):.1f} MB) -> 本機 SSD ...")
                shutil.copy2(src, dst)
        elif required:
            raise FileNotFoundError(
                f"Google Drive 缺少必備訓練檔：{src}\n"
                "請先手動將本機 tuner/data/hce/dataset_epd_wdl_20m.npz 上傳至 Drive 'MyDrive/chess_numba_data/hce/'。"
            )
        elif fname == "val_pgn.npz":
            print("[WARN] Drive 中未找到 val_pgn.npz；若下方要設 USE_PGN_VAL=True 請先手動上傳。")
        else:
            print(f"[Data] 選用檔 {fname} 未提供（未啟用該功能時可略過）。")

    print(f"\n✅ Colab + Google Drive 環境與資料就緒！即時備份目錄：{DRIVE_BACKUP_DIR}")
else:
    DRIVE_BACKUP_DIR = None
    print("✅ 偵測為本機環境，略過 Colab/Drive 掛載（直接使用本機 tuner/data/hce/）。")

In [ ]:
# 0. 環境檢查與載入 TexelTuningSession（首次載入約 25~45 秒；切換模式重跑本 Cell 僅需 ~3 秒）
import os
import sys
from pathlib import Path

# 自動定位專案根目錄（相容本機與 Colab）
if Path("chess_engine").is_dir() and Path("tuner").is_dir():
    REPO_ROOT = Path.cwd()
elif (Path.cwd().parent / "chess_engine").is_dir() and Path.cwd().name == "tuner":
    REPO_ROOT = Path.cwd().parent
    os.chdir(REPO_ROOT)
elif Path("/content/chess_numba").is_dir():
    REPO_ROOT = Path("/content/chess_numba")
    os.chdir(REPO_ROOT)
else:
    raise RuntimeError(f"請在專案根目錄啟動 Notebook，目前目錄：{Path.cwd()}")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tuner.notebook_session import TexelTuningSession

TRAIN_NPZ = REPO_ROOT / "tuner" / "data" / "hce" / "dataset_epd_wdl_20m.npz"
PGN_VAL_NPZ = REPO_ROOT / "tuner" / "data" / "hce" / "val_pgn.npz"
ZURICHESS_VAL_NPZ = REPO_ROOT / "tuner" / "data" / "hce" / "val_zurichess.npz"

# ==============================================================================
# 實驗模式參數（切換任一參數後直接重跑本 Cell，自動複用已載入的 20M 資料集，約 3 秒完成）
# ==============================================================================
# 1) 獨立驗證集選擇
USE_PGN_VAL = False            # True = 啟用 2M holdout + val_pgn.npz 雙重閘門；False = 僅用 2M unseen holdout
USE_ZURICHESS_VAL = False     # True = 額外啟用 val_zurichess.npz 第三平靜仲裁集（含 13.53% 易位權）

# 2) 訓練寫入目標引擎："classical"     -> chess_engine/classical/constants.py
#                    "classical_old" -> chess_engine/classical_old/constants.py
TARGET_ENGINE = "classical_old"

# 3) SF19 搜尋評估 + WDL 線性混合標籤：y_mix = (1 - SF_LAMBDA)*WDL + SF_LAMBDA*sigmoid(K_sf * sf_cp)
#    啟用前請先於終端機產生側車標籤檔（支援 Ctrl+C 續傳；可加 --limit 3000000 先跑快速前導實驗）：
#      python -m tuner.data_pipeline.annotate_sf_labels --nodes 4000 --skip-holdout
USE_SF_LABELS = False
SF_LAMBDA = 0.25              # 推薦 0.20 ~ 0.25（上限建議 <= 0.35；Unseen Holdout 與 val_pgn 永遠維持純 WDL）
SF_MAX_TACTICAL_GAP = 180     # |sf_cp - sf_d1_cp| > 180cp 的非平靜戰術局面自動排除出訓練梯度

# "auto": 若本機可用實體 RAM < 4.8 GB 自動取 10M train + 完整 1M dev + 2M holdout 防 Swap；設 None 強制載滿 17M train
MAX_TRAIN_ROWS = "auto"

if (
    "session" in globals()
    and getattr(session, "train_path", None) == TRAIN_NPZ
    and hasattr(session, "train_data")
):
    session.__class__ = TexelTuningSession
    session.reconfigure(
        use_pgn_val=USE_PGN_VAL,
        use_zurichess_val=USE_ZURICHESS_VAL,
        zurichess_val_npz=ZURICHESS_VAL_NPZ,
        target_engine=TARGET_ENGINE,
        pgn_val_npz=PGN_VAL_NPZ,
        use_sf_labels=USE_SF_LABELS,
        sf_lambda=SF_LAMBDA,
        sf_max_tactical_gap=SF_MAX_TACTICAL_GAP,
        drive_backup_dir=globals().get("DRIVE_BACKUP_DIR", None),
    )
else:
    session = TexelTuningSession(
        train_npz=TRAIN_NPZ,
        pgn_val_npz=PGN_VAL_NPZ,
        eval_sample=1_000_000,     # SPSA in-run dev / early-stop / K* 校準
        holdout_sample=2_000_000,  # 完全未見的訓練來源 holdout (永遠純 WDL)
        max_train_rows=MAX_TRAIN_ROWS,
        chunk_size=500_000,
        use_pgn_val=USE_PGN_VAL,
        use_zurichess_val=USE_ZURICHESS_VAL,
        zurichess_val_npz=ZURICHESS_VAL_NPZ,
        target_engine=TARGET_ENGINE,
        use_sf_labels=USE_SF_LABELS,
        sf_lambda=SF_LAMBDA,
        sf_max_tactical_gap=SF_MAX_TACTICAL_GAP,
        drive_backup_dir=globals().get("DRIVE_BACKUP_DIR", None),
    )
# 開啟「全自動無腦運行模式」：連續執行全部 Cell 時，僅當 Gate 判定為 PASS 才套用變更至目標 constants.py
session.require_pass_to_accept = True
print(
    f"[GUARD] 自動守衛已啟用：require_pass=True | target={session.target_engine!r} | "
    f"use_pgn_val={session.use_pgn_val} | use_sf_labels={session.use_sf_labels} "
    f"(lambda={session.sf_lambda:.2f}) | auto_K*={session._step_dev_k_star:.3f}"
)

Reconfigured session in 5.0s (reused preloaded dataset_epd_wdl_20m.npz):
  target_engine = 'classical_old' -> chess_engine\classical_old\constants.py
  use_pgn_val   = False | use_zurichess_val = False
  train_labels  = pure WDL | calibrated K* = 1.152
  runs_dir      = tuner\runs\classical_old_holdout_only
[GUARD] 自動守衛已啟用：require_pass=True | target='classical_old' | use_pgn_val=False | use_sf_labels=False (lambda=0.25) | auto_K*=1.152


In [52]:
# 0b. 檢視目前 Baseline 在兩個獨立驗證集上的健康度與最佳 K*
baseline_stats = session.baseline_report()

=== Current Step Baseline Health Report [target=classical_old -> chess_engine\classical_old\constants.py | use_pgn_val=False | train=pure_WDL | auto_K*=1.152] ===
  [1] dataset_epd_wdl_20m.npz unseen holdout [pure WDL] (n=2,000,000):
      MSE @ K=1.000 : 0.0924648
      MSE @ K*=1.152: 0.0921078
  [2] Independent PGN val: disabled (use_pgn_val=False)


In [53]:
# 0c. 熱重載調參器核心（若修改過 tuner/*.py，無須重啟 Kernel 或重載 20M 資料集即可生效）
import importlib
import tuner.data_pipeline.annotate_sf_labels as _sf_mod
import tuner.parameters as _params_mod
import tuner.tuner as _tuner_mod
import tuner.notebook_session as _ns_mod

importlib.reload(_sf_mod)
importlib.reload(_params_mod)
importlib.reload(_tuner_mod)
importlib.reload(_ns_mod)
if "session" in globals():
    session.__class__ = _ns_mod.TexelTuningSession
    session.require_pass_to_accept = True
print(
    "✅ 已熱重載最新 Scale-Aware SPSA、自動 K* 對齊、SF19 混合標籤與 Gate 判準！"
    + (
        f" [target={session.target_engine!r}, use_pgn_val={session.use_pgn_val}, "
        f"use_sf_labels={getattr(session, 'use_sf_labels', False)}]"
        if "session" in globals()
        else ""
    )
)

✅ 已熱重載最新 Scale-Aware SPSA、自動 K* 對齊、SF19 混合標籤與 Gate 判準！ [target='classical_old', use_pgn_val=False, use_sf_labels=False]


---
## Block 1：通路兵系統（Passed Pawns — 對應 `ORD-1..3` 與 `KP-DISCONT` 修復後最優先）
`ORD-1..3` 修正了通路兵路徑安全與攻擊遮罩計算順序，`KP-DISCONT` 補齊多兵純王兵殘局縮放：
- **Block 1a**：連續 centipawn 表（`PASSED_PAWN_BONUS`、`PASSED_FILE_BONUS`、`PASSED_CONNECTED_BONUS`、`PASSED_PHALANX_BONUS`），自動在基準校準 $K^*$ 下以 `alpha=65000` 調優，防止假性尺度膨脹。
- **Block 1b**：離散動態公式係數（`PASSED_DYNAMICS_MULT=3:5`、`PASSED_DYNAMICS_OFFSET=-12:-8` 置中於 `-10`、`KING_PROX_FRIENDLY_MULT=1:2`）窮舉掃描。
- **Block 1c**：路徑安全係數（`PASSED_PATH_*`，各為 $\theta_0 \pm 1$ 共 $3^4 = 81$ 組）改用精確離散網格窮舉，消除 SPSA 邊界截斷死鎖。
- **Block 1d**：殘局王距離權重與上下界（`KING_PROX_*`），以 `c=1.0, c_floor=1.0, alpha=50000` 對齊緊邊界整數格點。
- **Block 1e_grid**：多兵純王兵殘局近王活動倍率（`KING_PAWN_PROXIMITY_FACTOR=2:8`）離散窮舉掃描。

In [54]:
# [Run] Block 1a — 通路兵主獎勵表 (CP Tables, k_factor="auto" 對齊 K*)
res_1a = session.run_block(
    "block1a_passed_tables",
    tune=[
        "PASSED_PAWN_BONUS",
        "PASSED_FILE_BONUS",
        "PASSED_CONNECTED_BONUS",
        "PASSED_PHALANX_BONUS",
    ],
    iters=3500,
    c=2.5,
    alpha=65000.0,
    batch_size=65536,
    report_every=50,
    patience=25,
    seed=101,
)

=== Running Block [block1a_passed_tables] -> tuner\runs\classical_old_holdout_only\block1a_passed_tables.py (K=1.152, labels=WDL) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1203 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.4998, std=0.3841
  Continuous/mixed labels detected (102 unique values).
Freezing all parameters except: ['PASSED_PAWN_BONUS', 'PASSED_FILE_BONUS', 'PASSED_CONNECTED_BONUS', 'PASSED_PHALANX_BONUS']
Active parameters: 26 / 1272
Coordinate auto-scale active: min=1.00, median=1.00, max=4.00
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.152): 0.0923244
  Fixed-theta0 validation buckets (MSE, n):
    phase=0               0.0536540  n=74,346
    phase=1..31           0.0757888  n=38,267
    phase=32..63          0.0948313  n=33,258
    phase=64..9

In [55]:
# [Accept] Block 1a — 僅在 Gate 判定為 PASS 時自動套用，非 PASS 自動略過
session.accept_if_pass("block1a_passed_tables")


[PASS] [block1a_passed_tables] Gate 判定為 PASS -> 正在套用變更至 chess_engine\classical_old\constants.py 並建立備份...
  PASSED_PAWN_BONUS: [0, 0, 0, 14, 0, 20] -> [0, 0, 0, 14, 0, 20]
  PASSED_FILE_BONUS: [12, 5] -> [11, 6]
  PASSED_CONNECTED_BONUS: [0, 2, 5, 6, 10, 22] -> [0, 0, 3, 4, 8, 22]
  PASSED_PHALANX_BONUS: [0, 0, 6, 12, 25, 42] -> [0, 0, 6, 12, 24, 41]
Summary: 4 numeric change(s), 105 identical (skipped), 109 names in tuned dump.
Updated 4 constant(s) in chess_engine\classical_old\constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Updated chess_engine\classical_old\constants.py; removed 241 stale classical_old Numba cache file(s)
[block1a_passed_tables] Accepted 4 parameter group(s) into chess_engine\classical_old\constants.py. Backup saved to tuner\runs\classical_old_holdout_only\backups\20261108_181539_classical_old_before_block1a_passed_tables_constants.py.


['PASSED_PAWN_BONUS',
 'PASSED_FILE_BONUS',
 'PASSED_CONNECTED_BONUS',
 'PASSED_PHALANX_BONUS']

In [56]:
# [Run] Block 1b — 通路兵動態倍率離散窮舉 (置中於目前 MULT=4, OFFSET=-10)
res_1b = session.scan_grid(
    "block1b_passed_dynamics_grid",
    grids=[
        "PASSED_DYNAMICS_MULT=3:5",
        "PASSED_DYNAMICS_OFFSET=-12:-8",
        "KING_PROX_FRIENDLY_MULT=1:2",
    ],
    split="dev",
    eval_sample=1_000_000,
    top=10,
)

=== Discrete Grid Scan [block1b_passed_dynamics_grid] -> tuner\runs\classical_old_holdout_only\block1b_passed_dynamics_grid.py (K=1.161) ===
Scan sample (dev): 1,000,000 (K=1.161); excluded fixed validation=0
  evaluated   1/30: (3, -12, 1)
  evaluated   2/30: (3, -12, 2)
  evaluated   3/30: (3, -11, 1)
  evaluated   4/30: (3, -11, 2)
  evaluated   5/30: (3, -10, 1)
  evaluated   6/30: (3, -10, 2)
  evaluated   7/30: (3, -9, 1)
  evaluated   8/30: (3, -9, 2)
  evaluated   9/30: (3, -8, 1)
  evaluated  10/30: (3, -8, 2)
  evaluated  11/30: (4, -12, 1)
  evaluated  12/30: (4, -12, 2)
  evaluated  13/30: (4, -11, 1)
  evaluated  14/30: (4, -11, 2)
  evaluated  15/30: (4, -10, 1)
  evaluated  16/30: (4, -10, 2)
  evaluated  17/30: (4, -9, 1)
  evaluated  18/30: (4, -9, 2)
  evaluated  19/30: (4, -8, 1)
  evaluated  20/30: (4, -8, 2)
  evaluated  21/30: (5, -12, 1)
  evaluated  22/30: (5, -12, 2)
  evaluated  23/30: (5, -11, 1)
  evaluated  24/30: (5, -11, 2)
  evaluated  25/30: (5, -10, 1)

In [57]:
# [Accept] Block 1b — 若離散掃描在 holdout 與 val_pgn 皆穩定改善再套用
session.accept_if_pass("block1b_passed_dynamics_grid")


[SKIP] [block1b_passed_dynamics_grid] Gate 判定為 'NO-OP (0 integer changes)' (非 PASS) -> 自動跳過，保持既有 chess_engine\classical_old\constants.py 參數不變。


[]

In [58]:
# [Run] Block 1c — 通路兵路徑安全係數精確窮舉 (3^4 = 81 組，消除 SPSA ±1 邊界死鎖)
res_1c = session.scan_grid(
    "block1c_passed_path_grid",
    grids=[
        "PASSED_PATH_SAFE_NONE_ATTACK=9:11",
        "PASSED_PATH_SAFE_EDGE_ATTACK=5:7",
        "PASSED_PATH_SAFE_BLOCK_ONLY=1:3",
        "PASSED_PATH_SUPPORT_BONUS=2:4",
    ],
    split="dev",
    eval_sample=1_000_000,
    top=10,
)

=== Discrete Grid Scan [block1c_passed_path_grid] -> tuner\runs\classical_old_holdout_only\block1c_passed_path_grid.py (K=1.161) ===
Scan sample (dev): 1,000,000 (K=1.161); excluded fixed validation=0
  evaluated   1/81: (9, 5, 1, 2)
  evaluated   2/81: (9, 5, 1, 3)
  evaluated   3/81: (9, 5, 1, 4)
  evaluated   4/81: (9, 5, 2, 2)
  evaluated   5/81: (9, 5, 2, 3)
  evaluated   6/81: (9, 5, 2, 4)
  evaluated   7/81: (9, 5, 3, 2)
  evaluated   8/81: (9, 5, 3, 3)
  evaluated   9/81: (9, 5, 3, 4)
  evaluated  10/81: (9, 6, 1, 2)
  evaluated  11/81: (9, 6, 1, 3)
  evaluated  12/81: (9, 6, 1, 4)
  evaluated  13/81: (9, 6, 2, 2)
  evaluated  14/81: (9, 6, 2, 3)
  evaluated  15/81: (9, 6, 2, 4)
  evaluated  16/81: (9, 6, 3, 2)
  evaluated  17/81: (9, 6, 3, 3)
  evaluated  18/81: (9, 6, 3, 4)
  evaluated  19/81: (9, 7, 1, 2)
  evaluated  20/81: (9, 7, 1, 3)
  evaluated  21/81: (9, 7, 1, 4)
  evaluated  22/81: (9, 7, 2, 2)
  evaluated  23/81: (9, 7, 2, 3)
  evaluated  24/81: (9, 7, 2, 4)
  evalu

In [59]:
# [Accept] Block 1c
session.accept_if_pass("block1c_passed_path_grid")


[PASS] [block1c_passed_path_grid] Gate 判定為 PASS -> 正在套用變更至 chess_engine\classical_old\constants.py 並建立備份...
  PASSED_PATH_SAFE_NONE_ATTACK: [10] -> [9]
  PASSED_PATH_SAFE_EDGE_ATTACK: [6] -> [7]
  PASSED_PATH_SAFE_BLOCK_ONLY: [2] -> [3]
  PASSED_PATH_SUPPORT_BONUS: [3] -> [2]
Summary: 4 numeric change(s), 105 identical (skipped), 109 names in tuned dump.
Updated 4 constant(s) in chess_engine\classical_old\constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Updated chess_engine\classical_old\constants.py; removed 0 stale classical_old Numba cache file(s)
[block1c_passed_path_grid] Accepted 4 parameter group(s) into chess_engine\classical_old\constants.py. Backup saved to tuner\runs\classical_old_holdout_only\backups\20261108_181622_classical_old_before_block1c_passed_path_grid_constants.py.


['PASSED_PATH_SAFE_NONE_ATTACK',
 'PASSED_PATH_SAFE_EDGE_ATTACK',
 'PASSED_PATH_SAFE_BLOCK_ONLY',
 'PASSED_PATH_SUPPORT_BONUS']

In [60]:
# [Run] Block 1d — 通路兵王距離權重與夾限微調 (c=1.0 精確對齊 ±1 整數鄰域)
res_1d = session.run_block(
    "block1d_king_prox",
    tune=[
        "KING_PROXIMITY_MAX_DIST",
        "KING_PROX_ENEMY_MULT",
        "KING_PROX_ENEMY_DIV",
        "KING_PROX_MIN_BONUS",
        "KING_PROX_MAX_BONUS",
    ],
    iters=2500,
    c=1.0,
    c_floor=1.0,
    alpha=50000.0,
    batch_size=65536,
    report_every=50,
    patience=20,
    seed=104,
)

=== Running Block [block1d_king_prox] -> tuner\runs\classical_old_holdout_only\block1d_king_prox.py (K=1.164, labels=WDL) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1203 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.4998, std=0.3841
  Continuous/mixed labels detected (102 unique values).
Freezing all parameters except: ['KING_PROXIMITY_MAX_DIST', 'KING_PROX_ENEMY_MULT', 'KING_PROX_ENEMY_DIV', 'KING_PROX_MIN_BONUS', 'KING_PROX_MAX_BONUS']
Active parameters: 5 / 1272
Coordinate auto-scale active: min=1.00, median=1.00, max=2.70
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.164): 0.0922901
  Fixed-theta0 validation buckets (MSE, n):
    phase=0               0.0536078  n=74,346
    phase=1..31           0.0756214  n=38,267
    phase=32..63          0.0947671  n=33,

In [61]:
# [Accept] Block 1d
session.accept_if_pass("block1d_king_prox")


[PASS] [block1d_king_prox] Gate 判定為 PASS -> 正在套用變更至 chess_engine\classical_old\constants.py 並建立備份...
  KING_PROXIMITY_MAX_DIST: [7] -> [8]
  KING_PROX_MIN_BONUS: [-80] -> [-86]
  KING_PROX_MAX_BONUS: [81] -> [79]
Summary: 3 numeric change(s), 106 identical (skipped), 109 names in tuned dump.
Updated 3 constant(s) in chess_engine\classical_old\constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Updated chess_engine\classical_old\constants.py; removed 0 stale classical_old Numba cache file(s)
[block1d_king_prox] Accepted 3 parameter group(s) into chess_engine\classical_old\constants.py. Backup saved to tuner\runs\classical_old_holdout_only\backups\20261108_181856_classical_old_before_block1d_king_prox_constants.py.


['KING_PROXIMITY_MAX_DIST', 'KING_PROX_MIN_BONUS', 'KING_PROX_MAX_BONUS']

In [ ]:
# [Run] Block 1e_grid — 王兵純殘局近王活動係數離散窮舉
res_1e = session.scan_grid(
    "block1e_kp_prox_grid",
    grids=[
        "KING_PAWN_PROXIMITY_FACTOR=2:8",
    ],
    split="dev",
    eval_sample=1_000_000,
    top=10,
)

In [ ]:
# [Accept] Block 1e_grid
session.accept_if_pass("block1e_kp_prox_grid")

---
## Block 2：威脅、前哨與棋子協同（對應 `BUG-5` 威脅遮罩、`BUG-6` 弱后、`BUG-7` 前哨修復後）
- **Block 2a**：威脅評估表（`THREAT_*`，對王項已結構性釘死為 0）。
- **Block 2b**：騎士/主教前哨、車開放線、弱后、雙象與輕子協同（`OUTPOST_*`、`ROOK_ON_*`、`WEAK_QUEEN`、`BISHOP_*`、`KING_PROTECTOR`）。
- **Block 2b_grid**：主教同色兵遇到中心受阻兵時的懲罰加重倍率（`BISHOP_PAWNS_CENTER_BLOCKED_FACTOR=0:3`）離散窮舉掃描。

In [62]:
# [Run] Block 2a — 威脅表 (Threats, 調降 alpha=50000 防全域過衝)
res_2a = session.run_block(
    "block2a_threats",
    tune=[
        "THREAT_SAFE_PAWN",
        "THREAT_KNIGHT_ON_QUEEN",
        "THREAT_SLIDER_ON_QUEEN",
        "THREAT_BY_MINOR",
        "THREAT_BY_ROOK",
        "THREAT_BY_KING",
        "THREAT_HANGING",
        "THREAT_RESTRICTED_PIECE",
        "THREAT_PAWN_PUSH",
    ],
    iters=3500,
    c=2.5,
    alpha=50000.0,
    batch_size=65536,
    report_every=50,
    patience=25,
    seed=201,
)

=== Running Block [block2a_threats] -> tuner\runs\classical_old_holdout_only\block2a_threats.py (K=1.164, labels=WDL) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1203 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.4998, std=0.3841
  Continuous/mixed labels detected (102 unique values).
Freezing all parameters except: ['THREAT_SAFE_PAWN', 'THREAT_KNIGHT_ON_QUEEN', 'THREAT_SLIDER_ON_QUEEN', 'THREAT_BY_MINOR', 'THREAT_BY_ROOK', 'THREAT_BY_KING', 'THREAT_HANGING', 'THREAT_RESTRICTED_PIECE', 'THREAT_PAWN_PUSH']
Active parameters: 34 / 1272
Coordinate auto-scale active: min=1.00, median=1.00, max=2.43
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.164): 0.0922889
  Fixed-theta0 validation buckets (MSE, n):
    phase=0               0.0536080  n=74,346
    phase=1..31    

In [63]:
# [Accept] Block 2a
session.accept_if_pass("block2a_threats")


[PASS] [block2a_threats] Gate 判定為 PASS -> 正在套用變更至 chess_engine\classical_old\constants.py 並建立備份...
  THREAT_SAFE_PAWN: [58, 41] -> [55, 43]
  THREAT_KNIGHT_ON_QUEEN: [8, 0] -> [7, 0]
  THREAT_SLIDER_ON_QUEEN: [15, 0] -> [11, 0]
  THREAT_BY_MINOR: [6, 15, 24, 27, 42, 29] -> [4, 11, 25, 29, 40, 30]
  THREAT_BY_ROOK: [4, 22, 22, 28, 25, 25] -> [3, 18, 21, 26, 23, 23]
  THREAT_BY_KING: [18, 33] -> [19, 32]
  THREAT_HANGING: [4, 17] -> [2, 16]
  THREAT_RESTRICTED_PIECE: [0, 0] -> [2, 0]
  THREAT_PAWN_PUSH: [11, 14] -> [13, 15]
Summary: 9 numeric change(s), 100 identical (skipped), 109 names in tuned dump.
Updated 9 constant(s) in chess_engine\classical_old\constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Updated chess_engine\classical_old\constants.py; removed 0 stale classical_old Numba cache file(s)
[block2a_threats] Accepted 9 parameter group(s) into chess_engine\classical_old\constants.py. Backup saved to tuner\runs\cla

['THREAT_SAFE_PAWN',
 'THREAT_KNIGHT_ON_QUEEN',
 'THREAT_SLIDER_ON_QUEEN',
 'THREAT_BY_MINOR',
 'THREAT_BY_ROOK',
 'THREAT_BY_KING',
 'THREAT_HANGING',
 'THREAT_RESTRICTED_PIECE',
 'THREAT_PAWN_PUSH']

In [64]:
# [Run] Block 2b — 前哨與棋子協同 (Outposts & Piece Coordination)
res_2b = session.run_block(
    "block2b_outposts_coordination",
    tune=[
        "OUTPOST_BONUS_KNIGHT",
        "OUTPOST_BONUS_BISHOP",
        "REACHABLE_OUTPOST_BONUS",
        "MINOR_BEHIND_PAWN",
        "BISHOP_PAWNS_PENALTY",
        "LONG_DIAGONAL_BISHOP",
        "BISHOP_PAIR_BONUS",
        "BISHOP_PAIR_PAWN_SCALE",
        "ROOK_ON_SEMI_OPEN_FILE_BONUS",
        "ROOK_ON_OPEN_FILE_BONUS",
        "ROOK_ON_QUEEN_FILE",
        "WEAK_QUEEN",
        "KING_PROTECTOR",
    ],
    iters=3500,
    c=2.5,
    alpha=65000.0,
    batch_size=65536,
    report_every=50,
    patience=25,
    seed=202,
)

=== Running Block [block2b_outposts_coordination] -> tuner\runs\classical_old_holdout_only\block2b_outposts_coordination.py (K=1.169, labels=WDL) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1203 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.4998, std=0.3841
  Continuous/mixed labels detected (102 unique values).
Freezing all parameters except: ['OUTPOST_BONUS_KNIGHT', 'OUTPOST_BONUS_BISHOP', 'REACHABLE_OUTPOST_BONUS', 'MINOR_BEHIND_PAWN', 'BISHOP_PAWNS_PENALTY', 'LONG_DIAGONAL_BISHOP', 'BISHOP_PAIR_BONUS', 'BISHOP_PAIR_PAWN_SCALE', 'ROOK_ON_SEMI_OPEN_FILE_BONUS', 'ROOK_ON_OPEN_FILE_BONUS', 'ROOK_ON_QUEEN_FILE', 'WEAK_QUEEN', 'KING_PROTECTOR']
Active parameters: 42 / 1272
Coordinate auto-scale active: min=1.00, median=1.00, max=1.47
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initia

In [65]:
# [Accept] Block 2b
session.accept_if_pass("block2b_outposts_coordination")


[PASS] [block2b_outposts_coordination] Gate 判定為 PASS -> 正在套用變更至 chess_engine\classical_old\constants.py 並建立備份...
  BISHOP_PAIR_BONUS: [9, 14] -> [12, 12]
  BISHOP_PAIR_PAWN_SCALE: [17, 32] -> [16, 33]
  ROOK_ON_SEMI_OPEN_FILE_BONUS: [16, 2] -> [17, 1]
  ROOK_ON_OPEN_FILE_BONUS: [30, 18] -> [29, 15]
  OUTPOST_BONUS_KNIGHT: [0, 0, 0, 0, 12, 1] -> [0, 0, 0, 0, 11, 1]
  OUTPOST_BONUS_BISHOP: [0, 0, 0, 0, 4, 6] -> [0, 0, 0, 0, 4, 8]
  REACHABLE_OUTPOST_BONUS: [13, 6] -> [12, 6]
  MINOR_BEHIND_PAWN: [6, 19] -> [6, 18]
  ROOK_ON_QUEEN_FILE: [4, 3] -> [4, 2]
  WEAK_QUEEN: [21, 2] -> [21, 1]
Summary: 10 numeric change(s), 99 identical (skipped), 109 names in tuned dump.
Updated 10 constant(s) in chess_engine\classical_old\constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Updated chess_engine\classical_old\constants.py; removed 0 stale classical_old Numba cache file(s)
[block2b_outposts_coordination] Accepted 10 parameter group(s

['BISHOP_PAIR_BONUS',
 'BISHOP_PAIR_PAWN_SCALE',
 'ROOK_ON_SEMI_OPEN_FILE_BONUS',
 'ROOK_ON_OPEN_FILE_BONUS',
 'OUTPOST_BONUS_KNIGHT',
 'OUTPOST_BONUS_BISHOP',
 'REACHABLE_OUTPOST_BONUS',
 'MINOR_BEHIND_PAWN',
 'ROOK_ON_QUEEN_FILE',
 'WEAK_QUEEN']

In [ ]:
# [Run] Block 2b_grid — 主教同色兵中心受阻懲罰倍率離散掃描
res_2b_grid = session.scan_grid(
    "block2b_bishop_pawns_factor_grid",
    grids=[
        "BISHOP_PAWNS_CENTER_BLOCKED_FACTOR=0:3",
    ],
    split="dev",
    eval_sample=1_000_000,
    top=10,
)

In [ ]:
# [Accept] Block 2b_grid
session.accept_if_pass("block2b_bishop_pawns_factor_grid")

---
## Block 3：機動性查表（Mobility Tables — 對應 `BUG-1` 斜向釘死幾何修復後）
每張 Mobility 表的 `0-mobility` (MG/EG) 已錨定於 `theta0`（避免與棋子基礎材質產生常數偏移共線性），並強制單調遞增約束。

In [66]:
# [Run] Block 3 — 四子機動性查表 (N / B / R / Q Mobility)
res_3 = session.run_block(
    "block3_mobility",
    tune=[
        "KNIGHT_MOBILITY_BONUS",
        "BISHOP_MOBILITY_BONUS",
        "ROOK_MOBILITY_BONUS",
        "QUEEN_MOBILITY_BONUS",
    ],
    iters=4000,
    c=2.5,
    alpha=55000.0,
    batch_size=65536,
    report_every=50,
    patience=30,
    seed=301,
)

=== Running Block [block3_mobility] -> tuner\runs\classical_old_holdout_only\block3_mobility.py (K=1.171, labels=WDL) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1203 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.4998, std=0.3841
  Continuous/mixed labels detected (102 unique values).
Freezing all parameters except: ['KNIGHT_MOBILITY_BONUS', 'BISHOP_MOBILITY_BONUS', 'ROOK_MOBILITY_BONUS', 'QUEEN_MOBILITY_BONUS']
Active parameters: 124 / 1272
Coordinate auto-scale active: min=1.00, median=1.22, max=3.13
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.171): 0.0922051
  Fixed-theta0 validation buckets (MSE, n):
    phase=0               0.0535151  n=74,346
    phase=1..31           0.0754860  n=38,267
    phase=32..63          0.0946837  n=33,258
    phase=64..95     

In [67]:
# [Accept] Block 3
session.accept_if_pass("block3_mobility")


[PASS] [block3_mobility] Gate 判定為 PASS -> 正在套用變更至 chess_engine\classical_old\constants.py 並建立備份...
  KNIGHT_MOBILITY_BONUS: [-17, -26, -14, -20, 0, -8] -> [-17, -26, -12, -19, 2, -9]
  BISHOP_MOBILITY_BONUS: [-10, -18, 4, -13, 15, 1] -> [-10, -18, 5, -13, 15, 0]
  ROOK_MOBILITY_BONUS: [-27, -32, -4, -1, 10, 20] -> [-27, -32, -3, -1, 11, 19]
  QUEEN_MOBILITY_BONUS: [-26, -10, -14, -4, 7, 7] -> [-26, -10, -13, -5, 8, 7]
Summary: 4 numeric change(s), 105 identical (skipped), 109 names in tuned dump.
Updated 4 constant(s) in chess_engine\classical_old\constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Updated chess_engine\classical_old\constants.py; removed 0 stale classical_old Numba cache file(s)
[block3_mobility] Accepted 4 parameter group(s) into chess_engine\classical_old\constants.py. Backup saved to tuner\runs\classical_old_holdout_only\backups\20261108_183929_classical_old_before_block3_mobility_constants.py.


['KNIGHT_MOBILITY_BONUS',
 'BISHOP_MOBILITY_BONUS',
 'ROOK_MOBILITY_BONUS',
 'QUEEN_MOBILITY_BONUS']

---
## Block 4：兵型結構與連兵（Pawn Structure & Connected Pawns）
- **Block 4a**：孤兵、疊兵、落後兵、弱兵懲罰與連兵等級獎勵表。
- **Block 4b**：連兵支撐乘數（`CONNECTED_SUPPORT_WEIGHT`、`CONNECTED_SUPPORT_WEIGHT_EG`）離散窮舉。

In [68]:
# [Run] Block 4a — 兵型懲罰與連兵獎勵表
res_4a = session.run_block(
    "block4a_pawn_structure",
    tune=[
        "ISOLATED_PAWN_PENALTY",
        "DOUBLED_PAWN_PENALTY",
        "BACKWARD_PAWN_PENALTY",
        "WEAK_LEVER_PENALTY",
        "WEAK_UNOPPOSED_PENALTY",
        "CONNECTED_BONUS",
        "CONNECTED_BONUS_EG",
    ],
    iters=3500,
    c=2.5,
    alpha=60000.0,
    batch_size=65536,
    report_every=50,
    patience=25,
    seed=401,
)

=== Running Block [block4a_pawn_structure] -> tuner\runs\classical_old_holdout_only\block4a_pawn_structure.py (K=1.174, labels=WDL) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1203 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.4998, std=0.3841
  Continuous/mixed labels detected (102 unique values).
Freezing all parameters except: ['ISOLATED_PAWN_PENALTY', 'DOUBLED_PAWN_PENALTY', 'BACKWARD_PAWN_PENALTY', 'WEAK_LEVER_PENALTY', 'WEAK_UNOPPOSED_PENALTY', 'CONNECTED_BONUS', 'CONNECTED_BONUS_EG']
Active parameters: 22 / 1272
Coordinate auto-scale active: min=1.00, median=1.00, max=2.30
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.174): 0.0921800
  Fixed-theta0 validation buckets (MSE, n):
    phase=0               0.0534847  n=74,346
    phase=1..31           0.075454

In [69]:
# [Accept] Block 4a
session.accept_if_pass("block4a_pawn_structure")


[PASS] [block4a_pawn_structure] Gate 判定為 PASS -> 正在套用變更至 chess_engine\classical_old\constants.py 並建立備份...
  ISOLATED_PAWN_PENALTY: [0, -9] -> [0, -8]
  DOUBLED_PAWN_PENALTY: [-17, -32] -> [-15, -28]
  BACKWARD_PAWN_PENALTY: [0, -1] -> [0, 0]
  WEAK_UNOPPOSED_PENALTY: [-11, -3] -> [-10, -5]
  CONNECTED_BONUS: [0, 2, 7, 7, 7, 37] -> [0, 2, 7, 7, 7, 35]
  CONNECTED_BONUS_EG: [0, 0, 2, 12, 17, 22] -> [0, 4, 4, 11, 16, 21]
Summary: 6 numeric change(s), 103 identical (skipped), 109 names in tuned dump.
Updated 6 constant(s) in chess_engine\classical_old\constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Updated chess_engine\classical_old\constants.py; removed 0 stale classical_old Numba cache file(s)
[block4a_pawn_structure] Accepted 6 parameter group(s) into chess_engine\classical_old\constants.py. Backup saved to tuner\runs\classical_old_holdout_only\backups\20261108_184633_classical_old_before_block4a_pawn_structure_constan

['ISOLATED_PAWN_PENALTY',
 'DOUBLED_PAWN_PENALTY',
 'BACKWARD_PAWN_PENALTY',
 'WEAK_UNOPPOSED_PENALTY',
 'CONNECTED_BONUS',
 'CONNECTED_BONUS_EG']

In [70]:
# [Run] Block 4b — 連兵支撐權重離散窮舉 (置中於目前 MG=12, EG=7)
res_4b = session.scan_grid(
    "block4b_connected_support_grid",
    grids=[
        "CONNECTED_SUPPORT_WEIGHT=10:14",
        "CONNECTED_SUPPORT_WEIGHT_EG=5:9",
    ],
    split="dev",
    eval_sample=1_000_000,
    top=9,
)

=== Discrete Grid Scan [block4b_connected_support_grid] -> tuner\runs\classical_old_holdout_only\block4b_connected_support_grid.py (K=1.178) ===
Scan sample (dev): 1,000,000 (K=1.178); excluded fixed validation=0
  evaluated   1/25: (10, 5)
  evaluated   2/25: (10, 6)
  evaluated   3/25: (10, 7)
  evaluated   4/25: (10, 8)
  evaluated   5/25: (10, 9)
  evaluated   6/25: (11, 5)
  evaluated   7/25: (11, 6)
  evaluated   8/25: (11, 7)
  evaluated   9/25: (11, 8)
  evaluated  10/25: (11, 9)
  evaluated  11/25: (12, 5)
  evaluated  12/25: (12, 6)
  evaluated  13/25: (12, 7)
  evaluated  14/25: (12, 8)
  evaluated  15/25: (12, 9)
  evaluated  16/25: (13, 5)
  evaluated  17/25: (13, 6)
  evaluated  18/25: (13, 7)
  evaluated  19/25: (13, 8)
  evaluated  20/25: (13, 9)
  evaluated  21/25: (14, 5)
  evaluated  22/25: (14, 6)
  evaluated  23/25: (14, 7)
  evaluated  24/25: (14, 8)
  evaluated  25/25: (14, 9)
Candidates sorted by overall delta (negative is better):
  CONNECTED_SUPPORT_WEIGHT CON

In [71]:
# [Accept] Block 4b
session.accept_if_pass("block4b_connected_support_grid")


[PASS] [block4b_connected_support_grid] Gate 判定為 PASS -> 正在套用變更至 chess_engine\classical_old\constants.py 並建立備份...
  CONNECTED_SUPPORT_WEIGHT: [12] -> [14]
  CONNECTED_SUPPORT_WEIGHT_EG: [7] -> [5]
Summary: 2 numeric change(s), 107 identical (skipped), 109 names in tuned dump.
Updated 2 constant(s) in chess_engine\classical_old\constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Updated chess_engine\classical_old\constants.py; removed 0 stale classical_old Numba cache file(s)
[block4b_connected_support_grid] Accepted 2 parameter group(s) into chess_engine\classical_old\constants.py. Backup saved to tuner\runs\classical_old_holdout_only\backups\20261108_184649_classical_old_before_block4b_connected_support_grid_constants.py.


['CONNECTED_SUPPORT_WEIGHT', 'CONNECTED_SUPPORT_WEIGHT_EG']

---
## Block 5：王安全、王前兵盾與王 PST（對應 `BUG-8` King Ring 修復後）
依 `tuner/README.md` 規範，**將 King Danger 與 Shelter + 王 PST 分開調優**，防止三者搶同一組王安全訊號：
- **Block 5a**：`kingDanger` 連續權重（含 `KING_DANGER_NO_QUEEN` 與 `KING_DANGER_NO_QUEEN_ROOKLESS` 雙分流）。
- **Block 5a_grid**：`KING_DANGER_THRESHOLD` 與 `KING_DANGER_OFFSET` 離散閘門掃描。
- **Block 5a_flank_grid**：王翼側攻擊/防守與兵盾回饋整數係數（`KING_FLANK_ATTACK_NUM`、`KING_FLANK_DEFENSE_MULT`、`KING_SHELTER_FEEDBACK_NUM`，共 $5^3 = 125$ 組）離散掃描。
- **Block 5b**：`SHELTER_*`、`STORM`、`PAWNLESS_FLANK`、`FLANK_ATTACKS`、`TRAPPED_ROOK` 與窄帶王 `PST`（`king_pst_only=True`）。
- **Block 5c**：王安全輸出映射比例微調（`KING_DANGER_OUT_MG_NUM`、`KING_DANGER_OUT_EG_NUM`、`EG_SAFETY_SCALE`，依各自單位自動縮放後獨立調優，避免與 `5a/5b` 產生全域比例共線性）。
  > ⚠️ **特別檢查點**：執行 Block 5a/5b/5c 時，請務必確認 `val_pgn.npz` 的 **`castling!=0`** 桶（10.4 萬個真實具易位權局面）沒有顯著惡化！

In [72]:
# [Run] Block 5a — King Danger 攻擊權重與安全將軍係數 (auto_scale=6..15，調降 alpha=40000 防戰術過擬合)
res_5a = session.run_block(
    "block5a_king_danger",
    tune=[
        "KING_ATTACK_WEIGHTS",
        "SAFE_CHECK_QUEEN",
        "SAFE_CHECK_ROOK",
        "SAFE_CHECK_BISHOP",
        "SAFE_CHECK_KNIGHT",
        "KING_DANGER_WEAK_SQ",
        "KING_DANGER_UNSAFE_CHECK",
        "KING_DANGER_BLOCKERS",
        "KING_DANGER_ATTACK_ON_KING_SQ",
        "KING_DANGER_NO_QUEEN",
        "KING_DANGER_NO_QUEEN_ROOKLESS",
        "KING_DANGER_KNIGHT_DEF",
    ],
    iters=3500,
    c=2.5,
    alpha=40000.0,
    batch_size=65536,
    report_every=50,
    patience=25,
    seed=501,
)

=== Running Block [block5a_king_danger] -> tuner\runs\classical_old_holdout_only\block5a_king_danger.py (K=1.178, labels=WDL) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1203 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.4998, std=0.3841
  Continuous/mixed labels detected (102 unique values).
Freezing all parameters except: ['KING_ATTACK_WEIGHTS', 'SAFE_CHECK_QUEEN', 'SAFE_CHECK_ROOK', 'SAFE_CHECK_BISHOP', 'SAFE_CHECK_KNIGHT', 'KING_DANGER_WEAK_SQ', 'KING_DANGER_UNSAFE_CHECK', 'KING_DANGER_BLOCKERS', 'KING_DANGER_ATTACK_ON_KING_SQ', 'KING_DANGER_NO_QUEEN', 'KING_DANGER_NO_QUEEN_ROOKLESS', 'KING_DANGER_KNIGHT_DEF']
KING_ATTACK_WEIGHTS: keep NO_PIECE/PAWN frozen (indices 0,1)
Active parameters: 15 / 1272
Coordinate auto-scale active: min=1.00, median=3.03, max=14.25
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membe

In [73]:
# [Accept] Block 5a
session.accept_if_pass("block5a_king_danger")


[PASS] [block5a_king_danger] Gate 判定為 PASS -> 正在套用變更至 chess_engine\classical_old\constants.py 並建立備份...
  KING_ATTACK_WEIGHTS: [0, 0, 57, 19, 29, 18] -> [0, 0, 57, 19, 30, 20]
  SAFE_CHECK_QUEEN: [368] -> [355]
  SAFE_CHECK_ROOK: [477] -> [451]
  SAFE_CHECK_BISHOP: [376] -> [359]
  SAFE_CHECK_KNIGHT: [486] -> [464]
  KING_DANGER_WEAK_SQ: [150] -> [135]
  KING_DANGER_UNSAFE_CHECK: [91] -> [82]
  KING_DANGER_BLOCKERS: [47] -> [43]
  KING_DANGER_ATTACK_ON_KING_SQ: [43] -> [52]
  KING_DANGER_NO_QUEEN: [531] -> [539]
  KING_DANGER_NO_QUEEN_ROOKLESS: [855] -> [867]
  KING_DANGER_KNIGHT_DEF: [82] -> [77]
Summary: 12 numeric change(s), 97 identical (skipped), 109 names in tuned dump.
Updated 12 constant(s) in chess_engine\classical_old\constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Updated chess_engine\classical_old\constants.py; removed 0 stale classical_old Numba cache file(s)
[block5a_king_danger] Accepted 12 parameter gro

['KING_ATTACK_WEIGHTS',
 'SAFE_CHECK_QUEEN',
 'SAFE_CHECK_ROOK',
 'SAFE_CHECK_BISHOP',
 'SAFE_CHECK_KNIGHT',
 'KING_DANGER_WEAK_SQ',
 'KING_DANGER_UNSAFE_CHECK',
 'KING_DANGER_BLOCKERS',
 'KING_DANGER_ATTACK_ON_KING_SQ',
 'KING_DANGER_NO_QUEEN',
 'KING_DANGER_NO_QUEEN_ROOKLESS',
 'KING_DANGER_KNIGHT_DEF']

In [74]:
# [Run] Block 5a_grid — King Danger 啟動門檻與偏置離散掃描
res_5a_grid = session.scan_grid(
    "block5a_king_danger_grid",
    grids=[
        "KING_DANGER_OFFSET=0:4",
        "KING_DANGER_THRESHOLD=66:72",
    ],
    split="dev",
    eval_sample=1_000_000,
    top=10,
)

=== Discrete Grid Scan [block5a_king_danger_grid] -> tuner\runs\classical_old_holdout_only\block5a_king_danger_grid.py (K=1.183) ===
Scan sample (dev): 1,000,000 (K=1.183); excluded fixed validation=0
  evaluated   1/35: (0, 66)
  evaluated   2/35: (0, 67)
  evaluated   3/35: (0, 68)
  evaluated   4/35: (0, 69)
  evaluated   5/35: (0, 70)
  evaluated   6/35: (0, 71)
  evaluated   7/35: (0, 72)
  evaluated   8/35: (1, 66)
  evaluated   9/35: (1, 67)
  evaluated  10/35: (1, 68)
  evaluated  11/35: (1, 69)
  evaluated  12/35: (1, 70)
  evaluated  13/35: (1, 71)
  evaluated  14/35: (1, 72)
  evaluated  15/35: (2, 66)
  evaluated  16/35: (2, 67)
  evaluated  17/35: (2, 68)
  evaluated  18/35: (2, 69)
  evaluated  19/35: (2, 70)
  evaluated  20/35: (2, 71)
  evaluated  21/35: (2, 72)
  evaluated  22/35: (3, 66)
  evaluated  23/35: (3, 67)
  evaluated  24/35: (3, 68)
  evaluated  25/35: (3, 69)
  evaluated  26/35: (3, 70)
  evaluated  27/35: (3, 71)
  evaluated  28/35: (3, 72)
  evaluated  29

In [75]:
# [Accept] Block 5a_grid
session.accept_if_pass("block5a_king_danger_grid")


[SKIP] [block5a_king_danger_grid] Gate 判定為 'REJECT' (非 PASS) -> 自動跳過，保持既有 chess_engine\classical_old\constants.py 參數不變。


[]

In [ ]:
# [Run] Block 5a_flank_grid — 王翼攻防與兵盾回饋係數離散窮舉 (5^3 = 125 組)
res_5a_flank = session.scan_grid(
    "block5a_king_flank_grid",
    grids=[
        "KING_FLANK_ATTACK_NUM=1:5",
        "KING_FLANK_DEFENSE_MULT=2:6",
        "KING_SHELTER_FEEDBACK_NUM=4:8",
    ],
    split="dev",
    eval_sample=1_000_000,
    top=10,
)

In [ ]:
# [Accept] Block 5a_flank_grid
session.accept_if_pass("block5a_king_flank_grid")

In [76]:
session._reload_parameter_manager()
session._refresh_baseline_score_caches()

Updated chess_engine\classical_old\constants.py; removed 0 stale classical_old Numba cache file(s)


In [77]:
# [Run] Block 5b — 王前兵盾、兵風暴、受困車與窄帶王 PST (請重點檢查 val_pgn castling!=0 桶！)
res_5b = session.run_block(
    "block5b_shelter_storm_king_pst",
    tune=[
        "SHELTER_STRENGTH",
        "UNBLOCKED_STORM",
        "BLOCKED_STORM",
        "BLOCKED_STORM_EG",
        "SHELTER_BASE_MG",
        "SHELTER_BASE_EG",
        "KING_PAWN_DIST_PENALTY_EG",
        "PAWNLESS_FLANK",
        "FLANK_ATTACKS",
        "TRAPPED_ROOK",
        "PST_MG",
        "PST_EG",
    ],
    king_pst_only=True,
    iters=4000,
    c=2.5,
    alpha=55000.0,
    batch_size=98304,
    report_every=50,
    patience=30,
    seed=502,
)

=== Running Block [block5b_shelter_storm_king_pst] -> tuner\runs\classical_old_holdout_only\block5b_shelter_storm_king_pst.py (K=1.183, labels=WDL) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1203 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.4998, std=0.3841
  Continuous/mixed labels detected (102 unique values).
PST-piece mode active; also: ['SHELTER_STRENGTH', 'UNBLOCKED_STORM', 'BLOCKED_STORM', 'BLOCKED_STORM_EG', 'SHELTER_BASE_MG', 'SHELTER_BASE_EG', 'KING_PAWN_DIST_PENALTY_EG', 'PAWNLESS_FLANK', 'FLANK_ATTACKS', 'TRAPPED_ROOK']
PST piece filter: [K] rows active on PST_MG/EG
Active parameters: 195 / 1272
Coordinate auto-scale active: min=1.00, median=1.00, max=4.00
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.183): 0.0920933
  Fixed-theta0 validation bucket

In [78]:
# [Accept] Block 5b — 確認 val_pgn 的 all 與 castling!=0 桶皆無顯著退步後再套用
session.accept_if_pass("block5b_shelter_storm_king_pst")


[PASS] [block5b_shelter_storm_king_pst] Gate 判定為 PASS -> 正在套用變更至 chess_engine\classical_old\constants.py 並建立備份...
  PST_MG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
  PST_EG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
  TRAPPED_ROOK: [13, 1] -> [14, 0]
  SHELTER_STRENGTH: [13, 57, 62, 47, 33, 19] -> [15, 58, 60, 46, 31, 20]
  UNBLOCKED_STORM: [-69, 228, 108, -65, -42, -30] -> [-71, 236, 98, -61, -42, -31]
  BLOCKED_STORM: [-52] -> [-47]
  BLOCKED_STORM_EG: [-38] -> [-35]
  SHELTER_BASE_MG: [18] -> [19]
  SHELTER_BASE_EG: [9] -> [8]
  PAWNLESS_FLANK: [-10, -55] -> [-10, -57]
Summary: 10 numeric change(s), 99 identical (skipped), 109 names in tuned dump.
Updated 10 constant(s) in chess_engine\classical_old\constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Updated chess_engine\classical_old\constants.py; removed 0 stale classical_old Numba cache file(s)
[block5b_shelter_storm_king_pst] Accepted 10 parameter group(s) into che

['PST_MG',
 'PST_EG',
 'TRAPPED_ROOK',
 'SHELTER_STRENGTH',
 'UNBLOCKED_STORM',
 'BLOCKED_STORM',
 'BLOCKED_STORM_EG',
 'SHELTER_BASE_MG',
 'SHELTER_BASE_EG',
 'PAWNLESS_FLANK']

In [ ]:
# [Run] Block 5c — 王安全輸出尺度與殘局安全縮放 (auto_scale=3.3~512，防止與 5a/5b 共線漂移)
res_5c = session.run_block(
    "block5c_king_safety_output",
    tune=[
        "KING_DANGER_OUT_MG_NUM",
        "KING_DANGER_OUT_EG_NUM",
        "EG_SAFETY_SCALE",
    ],
    iters=2000,
    c=2.0,
    c_floor=1.25,
    alpha=35000.0,
    batch_size=65536,
    report_every=50,
    patience=20,
    seed=503,
)

In [ ]:
# [Accept] Block 5c
session.accept_if_pass("block5c_king_safety_output")

---
## Block 6：單棋種 PST 形狀精調（`N` → `B` → `R` → `Q` → `P`）
- `PST_MG` / `PST_EG` 每列固定 `theta0` 平均值（量化整數和守恆），本階段先獨立收斂各棋種的格間幾何形狀，子力基準價值留至 `Block 8` 與 `NBRQ PST` 聯合微調。
- 因單一棋子落在單格的出現率僅 `< 1% ~ 1.5%` 且 `P=128` 維 Rademacher 投影雜訊較大，`6a–6d` 放大 `batch_size=131072` 並提高 `alpha=90000~105000`，確保跨越整數格點。
- `Block 6e (Pawn)` 兵出現頻率高，使用 `batch_size=98304, alpha=35000` 穩健微調 Rank 2–7 形狀（`MG Pawn = 100` 永遠固定為全引擎錨點）。

In [79]:
# [Run] Block 6a_knight — 騎士 (Knight) PST 形狀調優 (batch=131072, alpha=90000)
res_6a_knight = session.run_block(
    "block6a_knight",
    tune=["PST_MG", "PST_EG"],
    pst_pieces="N",
    iters=4000,
    c=2.5,
    alpha=90000.0,
    batch_size=131072,
    report_every=50,
    patience=30,
    seed=601,
)

=== Running Block [block6a_knight] -> tuner\runs\classical_old_holdout_only\block6a_knight.py (K=1.184, labels=WDL) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1203 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.4998, std=0.3841
  Continuous/mixed labels detected (102 unique values).
PST-piece mode active
PST piece filter: [N] rows active on PST_MG/EG
Active parameters: 128 / 1272
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.184): 0.0920537
  Fixed-theta0 validation buckets (MSE, n):
    phase=0               0.0534116  n=74,346
    phase=1..31           0.0753386  n=38,267
    phase=32..63          0.0945405  n=33,258
    phase=64..95          0.1179707  n=54,949
    phase=96..128         0.1327841  n=49,180
    broad-EG phase<=64    0.0685785  n=146,006
    pie

In [80]:
# [Accept] Block 6a_knight
session.accept_if_pass("block6a_knight")


[PASS] [block6a_knight] Gate 判定為 PASS -> 正在套用變更至 chess_engine\classical_old\constants.py 並建立備份...
  PST_MG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
  PST_EG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
Summary: 2 numeric change(s), 107 identical (skipped), 109 names in tuned dump.
Updated 2 constant(s) in chess_engine\classical_old\constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Updated chess_engine\classical_old\constants.py; removed 0 stale classical_old Numba cache file(s)
[block6a_knight] Accepted 2 parameter group(s) into chess_engine\classical_old\constants.py. Backup saved to tuner\runs\classical_old_holdout_only\backups\20261108_192505_classical_old_before_block6a_knight_constants.py.


['PST_MG', 'PST_EG']

In [81]:
# [Run] Block 6b_bishop — 主教 (Bishop) PST 形狀調優 (batch=131072, alpha=90000)
res_6b_bishop = session.run_block(
    "block6b_bishop",
    tune=["PST_MG", "PST_EG"],
    pst_pieces="B",
    iters=4000,
    c=2.5,
    alpha=90000.0,
    batch_size=131072,
    report_every=50,
    patience=30,
    seed=602,
)

=== Running Block [block6b_bishop] -> tuner\runs\classical_old_holdout_only\block6b_bishop.py (K=1.184, labels=WDL) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1203 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.4998, std=0.3841
  Continuous/mixed labels detected (102 unique values).
PST-piece mode active
PST piece filter: [B] rows active on PST_MG/EG
Active parameters: 128 / 1272
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.184): 0.0920359
  Fixed-theta0 validation buckets (MSE, n):
    phase=0               0.0534053  n=74,346
    phase=1..31           0.0753314  n=38,267
    phase=32..63          0.0945201  n=33,258
    phase=64..95          0.1179506  n=54,949
    phase=96..128         0.1327384  n=49,180
    broad-EG phase<=64    0.0685687  n=146,006
    pie

In [82]:
# [Accept] Block 6b_bishop
session.accept_if_pass("block6b_bishop")


[PASS] [block6b_bishop] Gate 判定為 PASS -> 正在套用變更至 chess_engine\classical_old\constants.py 並建立備份...
  PST_MG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
  PST_EG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
Summary: 2 numeric change(s), 107 identical (skipped), 109 names in tuned dump.
Updated 2 constant(s) in chess_engine\classical_old\constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Updated chess_engine\classical_old\constants.py; removed 0 stale classical_old Numba cache file(s)
[block6b_bishop] Accepted 2 parameter group(s) into chess_engine\classical_old\constants.py. Backup saved to tuner\runs\classical_old_holdout_only\backups\20261108_194101_classical_old_before_block6b_bishop_constants.py.


['PST_MG', 'PST_EG']

In [83]:
# [Run] Block 6c_rook — 城堡 (Rook) PST 形狀調優 (batch=131072, alpha=95000)
res_6c_rook = session.run_block(
    "block6c_rook",
    tune=["PST_MG", "PST_EG"],
    pst_pieces="R",
    iters=4000,
    c=2.5,
    alpha=95000.0,
    batch_size=131072,
    report_every=50,
    patience=30,
    seed=603,
)

=== Running Block [block6c_rook] -> tuner\runs\classical_old_holdout_only\block6c_rook.py (K=1.185, labels=WDL) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1203 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.4998, std=0.3841
  Continuous/mixed labels detected (102 unique values).
PST-piece mode active
PST piece filter: [R] rows active on PST_MG/EG
Active parameters: 128 / 1272
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.185): 0.0920226
  Fixed-theta0 validation buckets (MSE, n):
    phase=0               0.0533949  n=74,346
    phase=1..31           0.0753291  n=38,267
    phase=32..63          0.0945111  n=33,258
    phase=64..95          0.1179426  n=54,949
    phase=96..128         0.1327252  n=49,180
    broad-EG phase<=64    0.0685607  n=146,006
    pieces=

In [84]:
# [Accept] Block 6c_rook
session.accept_if_pass("block6c_rook")


[PASS] [block6c_rook] Gate 判定為 PASS -> 正在套用變更至 chess_engine\classical_old\constants.py 並建立備份...
  PST_MG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
  PST_EG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
Summary: 2 numeric change(s), 107 identical (skipped), 109 names in tuned dump.
Updated 2 constant(s) in chess_engine\classical_old\constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Updated chess_engine\classical_old\constants.py; removed 0 stale classical_old Numba cache file(s)
[block6c_rook] Accepted 2 parameter group(s) into chess_engine\classical_old\constants.py. Backup saved to tuner\runs\classical_old_holdout_only\backups\20261108_200036_classical_old_before_block6c_rook_constants.py.


['PST_MG', 'PST_EG']

In [85]:
# [Run] Block 6d_queen — 皇后 (Queen) PST 形狀調優 (單格稀疏度最高，batch=131072, alpha=105000)
res_6d_queen = session.run_block(
    "block6d_queen",
    tune=["PST_MG", "PST_EG"],
    pst_pieces="Q",
    iters=4000,
    c=2.5,
    alpha=105000.0,
    batch_size=131072,
    report_every=50,
    patience=30,
    seed=604,
)

=== Running Block [block6d_queen] -> tuner\runs\classical_old_holdout_only\block6d_queen.py (K=1.186, labels=WDL) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1203 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.4998, std=0.3841
  Continuous/mixed labels detected (102 unique values).
PST-piece mode active
PST piece filter: [Q] rows active on PST_MG/EG
Active parameters: 128 / 1272
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.186): 0.0920019
  Fixed-theta0 validation buckets (MSE, n):
    phase=0               0.0533774  n=74,346
    phase=1..31           0.0752996  n=38,267
    phase=32..63          0.0944863  n=33,258
    phase=64..95          0.1179149  n=54,949
    phase=96..128         0.1327221  n=49,180
    broad-EG phase<=64    0.0685385  n=146,006
    piece

In [86]:
# [Accept] Block 6d_queen
session.accept_if_pass("block6d_queen")


[PASS] [block6d_queen] Gate 判定為 PASS -> 正在套用變更至 chess_engine\classical_old\constants.py 並建立備份...
  PST_MG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
  PST_EG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
Summary: 2 numeric change(s), 107 identical (skipped), 109 names in tuned dump.
Updated 2 constant(s) in chess_engine\classical_old\constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Updated chess_engine\classical_old\constants.py; removed 0 stale classical_old Numba cache file(s)
[block6d_queen] Accepted 2 parameter group(s) into chess_engine\classical_old\constants.py. Backup saved to tuner\runs\classical_old_holdout_only\backups\20261108_201405_classical_old_before_block6d_queen_constants.py.


['PST_MG', 'PST_EG']

In [87]:
# [Run] Block 6e_pawn — 士兵 PST 形狀調優 (調降 alpha=35000 防全域過衝)
res_6e_pawn = session.run_block(
    "block6e_pawn",
    tune=["PST_MG", "PST_EG"],
    pst_pieces="P",
    iters=3500,
    c=2.5,
    alpha=35000.0,
    batch_size=98304,
    report_every=50,
    patience=25,
    seed=605,
)

=== Running Block [block6e_pawn] -> tuner\runs\classical_old_holdout_only\block6e_pawn.py (K=1.187, labels=WDL) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1203 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.4998, std=0.3841
  Continuous/mixed labels detected (102 unique values).
PST-piece mode active
PST piece filter: [P] rows active on PST_MG/EG
Active parameters: 96 / 1272
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.187): 0.0919833
  Fixed-theta0 validation buckets (MSE, n):
    phase=0               0.0533720  n=74,346
    phase=1..31           0.0752875  n=38,267
    phase=32..63          0.0944640  n=33,258
    phase=64..95          0.1178921  n=54,949
    phase=96..128         0.1326912  n=49,180
    broad-EG phase<=64    0.0685276  n=146,006
    pieces=8

In [88]:
# [Accept] Block 6e_pawn
session.accept_if_pass("block6e_pawn")


[PASS] [block6e_pawn] Gate 判定為 PASS -> 正在套用變更至 chess_engine\classical_old\constants.py 並建立備份...
  PST_MG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
  PST_EG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
Summary: 2 numeric change(s), 107 identical (skipped), 109 names in tuned dump.
Updated 2 constant(s) in chess_engine\classical_old\constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Updated chess_engine\classical_old\constants.py; removed 0 stale classical_old Numba cache file(s)
[block6e_pawn] Accepted 2 parameter group(s) into chess_engine\classical_old\constants.py. Backup saved to tuner\runs\classical_old_holdout_only\backups\20261108_202321_classical_old_before_block6e_pawn_constants.py.


['PST_MG', 'PST_EG']

In [89]:
# session.rollback()

---
## Block 7：材質不平衡、主動權與空間/先手微調（Imbalance, Initiative, Space & Tempo）
- **Block 7a_div_grid**：材質不平衡總除數（`IMBALANCE_DIVISOR=20:22`）離散窮舉掃描（對齊 `abs_pm1` 區間）。
- **Block 7a**：二次材質不平衡矩陣（結構性 0 已自動釘死並排除出 SPSA mask，僅調非零交互項，±10% 軟帶）。
- **Block 7b**：Initiative 複雜度權重與偏置（未使用的 `INITIATIVE_PAWN_ENDGAME_WEIGHT_*` 已自動凍結）。
- **Block 7c_grid**：`TEMPO_BONUS` 與空間除數/門檻（`SPACE_*_DIVISOR`、`SPACE_THRESHOLD`）窮舉掃描。

In [ ]:
# [Run] Block 7a_div_grid — 材質不平衡除數離散掃描 (置中於 IMBALANCE_DIVISOR=21)
res_7a_div = session.scan_grid(
    "block7a_imbalance_div_grid",
    grids=[
        "IMBALANCE_DIVISOR=20:22",
    ],
    split="dev",
    eval_sample=1_000_000,
    top=10,
)

In [ ]:
# [Accept] Block 7a_div_grid
session.accept_if_pass("block7a_imbalance_div_grid")

In [90]:
# [Run] Block 7a — 二次材質不平衡矩陣 (Imbalance Polynomial)
res_7a = session.run_block(
    "block7a_imbalance",
    tune=[
        "IMBALANCE_QUADRATIC_OURS",
        "IMBALANCE_QUADRATIC_THEIRS",
        "IMBALANCE_SCALE_MG",
        "IMBALANCE_SCALE_EG",
    ],
    iters=3000,
    c=2.0,
    alpha=45000.0,
    batch_size=65536,
    report_every=50,
    patience=25,
    seed=701,
)

=== Running Block [block7a_imbalance] -> tuner\runs\classical_old_holdout_only\block7a_imbalance.py (K=1.189, labels=WDL) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1203 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.4998, std=0.3841
  Continuous/mixed labels detected (102 unique values).
Freezing all parameters except: ['IMBALANCE_QUADRATIC_OURS', 'IMBALANCE_QUADRATIC_THEIRS', 'IMBALANCE_SCALE_MG', 'IMBALANCE_SCALE_EG']
IMBALANCE_QUADRATIC_OURS: froze 17 theta0-zero structural/stability cells
IMBALANCE_QUADRATIC_THEIRS: froze 21 theta0-zero structural/stability cells
Active parameters: 36 / 1272
Coordinate auto-scale active: min=1.00, median=2.27, max=20.00
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.189): 0.0919343
  Fixed-theta0 validation buckets (MSE, n):


In [91]:
# [Accept] Block 7a
session.accept_if_pass("block7a_imbalance")


[PASS] [block7a_imbalance] Gate 判定為 PASS -> 正在套用變更至 chess_engine\classical_old\constants.py 並建立備份...
  IMBALANCE_QUADRATIC_OURS: [1353, 0, 0, 0, 0, 0] -> [1299, 0, 0, 0, 0, 0]
  IMBALANCE_QUADRATIC_THEIRS: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
  IMBALANCE_SCALE_MG: [53] -> [54]
  IMBALANCE_SCALE_EG: [68] -> [73]
Summary: 4 numeric change(s), 105 identical (skipped), 109 names in tuned dump.
Updated 4 constant(s) in chess_engine\classical_old\constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Updated chess_engine\classical_old\constants.py; removed 0 stale classical_old Numba cache file(s)
[block7a_imbalance] Accepted 4 parameter group(s) into chess_engine\classical_old\constants.py. Backup saved to tuner\runs\classical_old_holdout_only\backups\20261108_203006_classical_old_before_block7a_imbalance_constants.py.


['IMBALANCE_QUADRATIC_OURS',
 'IMBALANCE_QUADRATIC_THEIRS',
 'IMBALANCE_SCALE_MG',
 'IMBALANCE_SCALE_EG']

In [92]:
# [Run] Block 7b — 主動權複雜度係數 (Initiative / Complexity, 對齊 K* 消除尺度膨脹)
res_7b = session.run_block(
    "block7b_initiative",
    tune=[
        "INITIATIVE_PASSED_WEIGHT_MG",
        "INITIATIVE_PAWN_WEIGHT_MG",
        "INITIATIVE_OUTFLANKING_WEIGHT_MG",
        "INITIATIVE_INFILTRATION_WEIGHT_MG",
        "INITIATIVE_BOTH_FLANKS_WEIGHT_MG",
        "INITIATIVE_ALMOST_UNWIN_WEIGHT_MG",
        "INITIATIVE_OFFSET_MG",
        "INITIATIVE_MG_OFFSET",
        "INITIATIVE_PASSED_WEIGHT_EG",
        "INITIATIVE_PAWN_WEIGHT_EG",
        "INITIATIVE_OUTFLANKING_WEIGHT_EG",
        "INITIATIVE_INFILTRATION_WEIGHT_EG",
        "INITIATIVE_BOTH_FLANKS_WEIGHT_EG",
        "INITIATIVE_ALMOST_UNWIN_WEIGHT_EG",
        "INITIATIVE_OFFSET_EG",
    ],
    iters=2500,
    c=1.5,
    c_floor=1.25,
    alpha=30000.0,
    batch_size=65536,
    report_every=50,
    patience=20,
    seed=702,
)

=== Running Block [block7b_initiative] -> tuner\runs\classical_old_holdout_only\block7b_initiative.py (K=1.193, labels=WDL) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1203 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.4998, std=0.3841
  Continuous/mixed labels detected (102 unique values).
Freezing all parameters except: ['INITIATIVE_PASSED_WEIGHT_MG', 'INITIATIVE_PAWN_WEIGHT_MG', 'INITIATIVE_OUTFLANKING_WEIGHT_MG', 'INITIATIVE_INFILTRATION_WEIGHT_MG', 'INITIATIVE_BOTH_FLANKS_WEIGHT_MG', 'INITIATIVE_ALMOST_UNWIN_WEIGHT_MG', 'INITIATIVE_OFFSET_MG', 'INITIATIVE_MG_OFFSET', 'INITIATIVE_PASSED_WEIGHT_EG', 'INITIATIVE_PAWN_WEIGHT_EG', 'INITIATIVE_OUTFLANKING_WEIGHT_EG', 'INITIATIVE_INFILTRATION_WEIGHT_EG', 'INITIATIVE_BOTH_FLANKS_WEIGHT_EG', 'INITIATIVE_ALMOST_UNWIN_WEIGHT_EG', 'INITIATIVE_OFFSET_EG']
Active parameters: 15 / 1272
Coordinate auto-scale active: min=1.00, median=1.00, max=2.57
Dev val: 1,000,000

In [93]:
# [Accept] Block 7b
session.accept_if_pass("block7b_initiative")


[PASS] [block7b_initiative] Gate 判定為 PASS -> 正在套用變更至 chess_engine\classical_old\constants.py 並建立備份...
  INITIATIVE_PAWN_WEIGHT_EG: [5] -> [4]
Summary: 1 numeric change(s), 108 identical (skipped), 109 names in tuned dump.
Updated 1 constant(s) in chess_engine\classical_old\constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Updated chess_engine\classical_old\constants.py; removed 0 stale classical_old Numba cache file(s)
[block7b_initiative] Accepted 1 parameter group(s) into chess_engine\classical_old\constants.py. Backup saved to tuner\runs\classical_old_holdout_only\backups\20261108_203230_classical_old_before_block7b_initiative_constants.py.


['INITIATIVE_PAWN_WEIGHT_EG']

In [94]:
# [Run] Block 7c_grid — Tempo、Space 門檻與除數離散掃描 (置中於 SPACE_THRESHOLD=6000)
res_7c = session.scan_grid(
    "block7c_space_tempo_grid",
    grids=[
        "SPACE_BONUS_DIVISOR=14:16",
        "SPACE_SCALE_DIVISOR=5:7",
        "SPACE_THRESHOLD=5800:6200:100",
        "TEMPO_BONUS=18:20",
    ],
    split="dev",
    eval_sample=1_000_000,
    top=10,
)

=== Discrete Grid Scan [block7c_space_tempo_grid] -> tuner\runs\classical_old_holdout_only\block7c_space_tempo_grid.py (K=1.236) ===
Scan sample (dev): 1,000,000 (K=1.236); excluded fixed validation=0
  evaluated   1/135: (14, 5, 5800, 18)
  evaluated   2/135: (14, 5, 5800, 19)
  evaluated   3/135: (14, 5, 5800, 20)
  evaluated   4/135: (14, 5, 5900, 18)
  evaluated   5/135: (14, 5, 5900, 19)
  evaluated   6/135: (14, 5, 5900, 20)
  evaluated   7/135: (14, 5, 6000, 18)
  evaluated   8/135: (14, 5, 6000, 19)
  evaluated   9/135: (14, 5, 6000, 20)
  evaluated  10/135: (14, 5, 6100, 18)
  evaluated  11/135: (14, 5, 6100, 19)
  evaluated  12/135: (14, 5, 6100, 20)
  evaluated  13/135: (14, 5, 6200, 18)
  evaluated  14/135: (14, 5, 6200, 19)
  evaluated  15/135: (14, 5, 6200, 20)
  evaluated  16/135: (14, 6, 5800, 18)
  evaluated  17/135: (14, 6, 5800, 19)
  evaluated  18/135: (14, 6, 5800, 20)
  evaluated  19/135: (14, 6, 5900, 18)
  evaluated  20/135: (14, 6, 5900, 19)
  evaluated  21/135

In [95]:
# [Accept] Block 7c_grid — 可用 only=[...] 只挑確認有效的常數套用
session.accept_if_pass("block7c_space_tempo_grid")


[PASS] [block7c_space_tempo_grid] Gate 判定為 PASS -> 正在套用變更至 chess_engine\classical_old\constants.py 並建立備份...
  SPACE_BONUS_DIVISOR: [15] -> [16]
  SPACE_SCALE_DIVISOR: [6] -> [7]
  TEMPO_BONUS: [19] -> [20]
Summary: 3 numeric change(s), 106 identical (skipped), 109 names in tuned dump.
Updated 3 constant(s) in chess_engine\classical_old\constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Updated chess_engine\classical_old\constants.py; removed 0 stale classical_old Numba cache file(s)
[block7c_space_tempo_grid] Accepted 3 parameter group(s) into chess_engine\classical_old\constants.py. Backup saved to tuner\runs\classical_old_holdout_only\backups\20261108_203308_classical_old_before_block7c_space_tempo_grid_constants.py.


['SPACE_BONUS_DIVISOR', 'SPACE_SCALE_DIVISOR', 'TEMPO_BONUS']

---
## Block 8：全子力聯合收尾 Polish（`NBRQ` Material + PST Joint Polish）
在前面各獨立區塊已收斂並接受後，以 `batch_size=131072, alpha=40000, c=2.0, iters=4500` 同時打開 `N/B/R/Q` 的材質與 PST（共 520 自由度），消除跨區塊之間的殘餘交互作用。

In [96]:
# [Run] Block 8 — NBRQ Material + PST 聯合 Polish (520 參數，batch=131072, alpha=40000)
res_8 = session.run_block(
    "block8_nbrq_joint_polish",
    tune=["MG_MATERIAL_VALUES", "EG_MATERIAL_VALUES", "PST_MG", "PST_EG"],
    pst_pieces="NBRQ",
    iters=4500,
    c=2.0,
    alpha=40000.0,
    batch_size=131072,
    report_every=50,
    patience=35,
    seed=801,
)

=== Running Block [block8_nbrq_joint_polish] -> tuner\runs\classical_old_holdout_only\block8_nbrq_joint_polish.py (K=1.236, labels=WDL) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1203 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.4998, std=0.3841
  Continuous/mixed labels detected (102 unique values).
PST-piece mode active; also: ['MG_MATERIAL_VALUES', 'EG_MATERIAL_VALUES']
PST piece filter: [N,B,R,Q] rows active on PST_MG/EG
MG_MATERIAL_VALUES: keep Pawn/King frozen (indices 0,5)
EG_MATERIAL_VALUES: keep King frozen (index 5)
Active parameters: 520 / 1272
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.236): 0.0918372
  Fixed-theta0 validation buckets (MSE, n):
    phase=0               0.0531592  n=74,346
    phase=1..31           0.0749608  n=38,267
    phase=3

In [97]:
# [Accept] Block 8
session.accept_if_pass("block8_nbrq_joint_polish")


[PASS] [block8_nbrq_joint_polish] Gate 判定為 PASS -> 正在套用變更至 chess_engine\classical_old\constants.py 並建立備份...
  MG_MATERIAL_VALUES: [100, 350, 363, 572, 1172, 0] -> [100, 352, 361, 570, 1174, 0]
  EG_MATERIAL_VALUES: [117, 420, 440, 696, 1291, 0] -> [117, 419, 440, 697, 1295, 0]
  PST_MG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
  PST_EG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
Summary: 4 numeric change(s), 105 identical (skipped), 109 names in tuned dump.
Updated 4 constant(s) in chess_engine\classical_old\constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Updated chess_engine\classical_old\constants.py; removed 0 stale classical_old Numba cache file(s)
[block8_nbrq_joint_polish] Accepted 4 parameter group(s) into chess_engine\classical_old\constants.py. Backup saved to tuner\runs\classical_old_holdout_only\backups\20261108_204828_classical_old_before_block8_nbrq_joint_polish_constants.py.


['MG_MATERIAL_VALUES', 'EG_MATERIAL_VALUES', 'PST_MG', 'PST_EG']

---
## 9. 總結驗收、回滾工具與分階段對戰流程
完成一輪調參後：
1. 執行下方 **`session.cumulative_report()`** 檢視相較於本次啟動前初始 `theta0` 的累積變動與驗證集總增益。
2. 執行 **Cell `9c`** 依 `session.target_engine` 同步快取並通過一致性與殘局不變量單元測試。
3. **三階段實驗路線**：
   - **階段一（新超參數 + 雙重驗證 vs 現有基準）**：Cell 1 設 `USE_PGN_VAL = True, USE_SF_LABELS = False, TARGET_ENGINE = "classical"` 跑完全流程 → 對戰 `classical_old`：
     `python tools/tournament.py -c 4 --games 500 --nodes 300000 --name1 New_WithPGN --name2 Old_Base`
   - **階段二（驗證集模式 A/B 對戰）**：若階段一勝出，經同意同步基準後，Cell 1 改設 `USE_PGN_VAL = False, TARGET_ENGINE = "classical"` 重跑一輪 → 對戰驗證 `HoldoutOnly` vs `WithPGN`。
   - **階段三（SF19 搜尋 + WDL 混合標籤升級）**：先於終端機執行 `python -m tuner.data_pipeline.annotate_sf_labels --nodes 4000 --skip-holdout`，完成後於 Cell 1 設 `USE_SF_LABELS = True, SF_LAMBDA = 0.25` 重跑 Cell 1 接續精調並對戰驗收。

In [98]:
# 9a. 累積總成績單（目前 constants.py 對比本次 Session 啟動前的初始 theta0）
cum_stats = session.cumulative_report()

=== Cumulative Progress vs Initial Session Baseline (theta0) [target=classical_old -> chess_engine\classical_old\constants.py | use_pgn_val=False] ===
Effective integer changes:
  MG_MATERIAL_VALUES: changed=4/6, L1=8
  EG_MATERIAL_VALUES: changed=3/6, L1=6
  PST_MG: changed=200/384, L1=296
  PST_EG: changed=212/384, L1=312
  KNIGHT_MOBILITY_BONUS: changed=14/18, L1=18
  BISHOP_MOBILITY_BONUS: changed=17/28, L1=20
  ROOK_MOBILITY_BONUS: changed=22/30, L1=28
  QUEEN_MOBILITY_BONUS: changed=41/56, L1=70
  BISHOP_PAIR_BONUS: changed=2/2, L1=5
  BISHOP_PAIR_PAWN_SCALE: changed=2/2, L1=2
  ROOK_ON_SEMI_OPEN_FILE_BONUS: changed=2/2, L1=2
  ROOK_ON_OPEN_FILE_BONUS: changed=2/2, L1=4
  PASSED_PAWN_BONUS: changed=6/16, L1=28
  PASSED_FILE_BONUS: changed=2/2, L1=2
  PASSED_CONNECTED_BONUS: changed=4/8, L1=8
  PASSED_PHALANX_BONUS: changed=3/8, L1=4
  PASSED_PATH_SAFE_NONE_ATTACK: 10 -> 9
  PASSED_PATH_SAFE_EDGE_ATTACK: 6 -> 7
  PASSED_PATH_SAFE_BLOCK_ONLY: 2 -> 3
  PASSED_PATH_SUPPORT_BONUS: 3 -

In [99]:
# 9b. 【選用工具】回滾上一步 或 手動覆寫單一參數重新驗證
# 若要撤銷最近一次 accept()：
# session.rollback()

# 若要撤銷到特定區塊之前的備份（例如撤銷 block5b 及其之後的 accept）：
# session.rollback("block5b_shelter_storm_king_pst")

# 若要測試某個候選檔加上手動微調單一值：
# session.validate_file("block5a_king_danger", overrides=["KING_DANGER_NO_QUEEN=570"])

In [100]:
# 9c. 依 target_engine 同步快取並執行評估一致性與殘局不變量單元測試
import subprocess

target_eng = getattr(session, "target_engine", "classical")
if target_eng == "classical":
    cmds = [
        [sys.executable, "-m", "tuner.sync_constants_snapshot"],
        [sys.executable, "-m", "tuner.codegen.generate_tunable_eval"],
        [
            sys.executable, "-m", "unittest",
            "tests.test_texel_parameterization",
            "tests.test_endgame_conformance",
            "tests.test_tunable_eval_parity",
            "tests.test_runtime_eval_weights",
            "-v",
        ],
    ]
else:
    parity_snippet = (
        "from pathlib import Path; "
        "from chess_engine.classical.fen_parser import parse_fen; "
        "from chess_engine.classical_old.evaluation import evaluate_position as eval_old; "
        "from tuner.parameters import ParameterManager; "
        "from tuner.tunable_eval import evaluate_position_tunable; "
        "from tuner.sync_constants_snapshot import clear_classical_numba_cache; "
        "clear_classical_numba_cache(Path('chess_engine/classical_old/__pycache__')); "
        "theta = ParameterManager('chess_engine/classical_old/constants.py').get_initial_theta(); "
        "fens = ["
        "'rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1',"
        "'r3k2r/p1ppqpb1/bn2pnp1/3PN3/1p2P3/2N2Q1p/PPPBBPPP/R3K2R w KQkq - 0 1',"
        "'8/2p5/3p4/KP5r/1R3p1k/8/4P1P1/8 w - - 0 1'"
        "]; "
        "diffs = []; "
        "[diffs.append((f, int(eval_old(*parse_fen(f), lazy=False)[0]), int(evaluate_position_tunable(*parse_fen(f), theta, False)))) for f in fens]; "
        "bad = [d for d in diffs if d[1] != d[2]]; "
        "assert not bad, f'classical_old parity mismatch: {bad}'; "
        "print('classical_old parity OK:', [(d[1], d[2]) for d in diffs])"
    )
    cmds = [
        [sys.executable, "-m", "unittest", "tests.test_texel_parameterization", "-v"],
        [sys.executable, "-c", parity_snippet],
    ]

for cmd in cmds:
    print("+", " ".join(str(x) for x in cmd[:4]), "..." if len(cmd) > 4 else "")
    rc = subprocess.call(cmd, cwd=str(REPO_ROOT))
    if rc != 0:
        raise RuntimeError(f"Command failed with exit code {rc}: {cmd}")
print(f"✅ [{target_eng}] 一致性與參數化測試通過！可進行對局閘門驗證：")
print("   python tools/tournament.py -c 4 --games 500 --nodes 300000 --name1 WithPGN --name2 HoldoutOnly")

+ c:\Users\ren cian\AppData\Local\Programs\Python\Python310\python.exe -m unittest tests.test_texel_parameterization ...
+ c:\Users\ren cian\AppData\Local\Programs\Python\Python310\python.exe -c from pathlib import Path; from chess_engine.classical.fen_parser import parse_fen; from chess_engine.classical_old.evaluation import evaluate_position as eval_old; from tuner.parameters import ParameterManager; from tuner.tunable_eval import evaluate_position_tunable; from tuner.sync_constants_snapshot import clear_classical_numba_cache; clear_classical_numba_cache(Path('chess_engine/classical_old/__pycache__')); theta = ParameterManager('chess_engine/classical_old/constants.py').get_initial_theta(); fens = ['rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1','r3k2r/p1ppqpb1/bn2pnp1/3PN3/1p2P3/2N2Q1p/PPPBBPPP/R3K2R w KQkq - 0 1','8/2p5/3p4/KP5r/1R3p1k/8/4P1P1/8 w - - 0 1']; diffs = []; [diffs.append((f, int(eval_old(*parse_fen(f), lazy=False)[0]), int(evaluate_position_tunable(*parse_fen

In [ ]:
# 9d. 【Colab / Google Drive】完整同步與打包下載最新調參結果
from datetime import datetime
import zipfile

target_eng = getattr(session, "target_engine", "classical")
target_const = Path(getattr(session, "target_constants_path", REPO_ROOT / "chess_engine" / target_eng / "constants.py"))
runs_dir = Path(getattr(session, "runs_dir", REPO_ROOT / "tuner" / "runs"))

session.sync_to_drive(tag="final")
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_name = f"texel_tuned_{target_eng}_{runs_dir.name}_{ts}.zip"

out_root = globals().get("DRIVE_BACKUP_DIR") or runs_dir
out_root = Path(out_root)
out_root.mkdir(parents=True, exist_ok=True)
zip_path = out_root / zip_name

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    if target_const.is_file():
        zf.write(target_const, arcname=f"chess_engine/{target_eng}/constants.py")
    for py_file in sorted(runs_dir.glob("*.py")):
        zf.write(py_file, arcname=f"tuner/runs/{runs_dir.name}/{py_file.name}")
    snap = REPO_ROOT / "tuner" / "constants_snapshot.py"
    if target_eng == "classical" and snap.is_file():
        zf.write(snap, arcname="tuner/constants_snapshot.py")

print(f"📦 已匯出調參成果壓縮包：{zip_path}")
if target_const.is_file():
    print(f"📄 目標引擎常數檔：{target_const}")
if globals().get("DRIVE_BACKUP_DIR"):
    print(f"☁️ Google Drive 已保留未壓縮常數檔：{Path(DRIVE_BACKUP_DIR) / (target_eng + '_constants.py')}")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass